Menyiapkan DataSet

In [2]:
import numpy as np
import pandas as pd

# Tabel 1: Target & PIC per cabang kota (tabel referensi, dibuat langsung sebagai DataFrame)
data_target_cabang = {
    "kota": ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"],
    "target_bulanan": [45000000, 60000000, 55000000, 40000000, 30000000],
    "pic_cabang": ["Rani", "Joko", "Sari", "Bayu", "Fitri"],
}

# Tabel 2: Data transaksi (disimpan sebagai CSV, lalu diunggah ke HDFS)
np.random.seed(55)
n = 500
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"]
data_transaksi_t5 = {
    "order_id": [f"TRX-{i}" for i in range(n)],
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 10, size=n),
    "harga_satuan": np.random.choice([25000, 50000, 75000, 100000, 150000], size=n),
}
pd.DataFrame(data_transaksi_t5).to_csv("transaksi_tugas5.csv", index=False)

!hdfs dfs -mkdir -p /user/mahasiswa/tugas5
!hdfs dfs -put -f transaksi_tugas5.csv /user/mahasiswa/tugas5/
print("Dataset siap. Tabel transaksi sudah diunggah ke HDFS: /user/mahasiswa/tugas5/transaksi_tugas5.csv")
print("Simpan juga 'data_target_cabang' di atas — kalian akan membuatnya menjadi DataFrame sendiri di notebook tugas.")

Dataset siap. Tabel transaksi sudah diunggah ke HDFS: /user/mahasiswa/tugas5/transaksi_tugas5.csv
Simpan juga 'data_target_cabang' di atas — kalian akan membuatnya menjadi DataFrame sendiri di notebook tugas.


In [6]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
import pandas as pd

# Inisiasi SparkSession
spark = SparkSession.builder.appName("Tugas5").master("local[*]").getOrCreate()

# 1. Membaca data transaksi dari HDFS dan menambahkan kolom pendapatan
df_transaksi = spark.read.csv("hdfs://localhost:9000/user/mahasiswa/tugas5/transaksi_tugas5.csv", header=True, inferSchema=True)
df_transaksi = df_transaksi.withColumn("pendapatan", col("unit_terjual") * col("harga_satuan"))

# 2. Membuat df_target dari dictionary
data_target_cabang = {
    "kota": ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"],
    "target_bulanan": [45000000, 60000000, 55000000, 40000000, 30000000],
    "pic_cabang": ["Rani", "Joko", "Sari", "Bayu", "Fitri"],
}
df_target = spark.createDataFrame(pd.DataFrame(data_target_cabang))

In [7]:
# A. Join & Perbandingan Target

from pyspark.sql.functions import sum as spark_sum

# Meringkas total pendapatan per kota
ringkasan_kota = df_transaksi.groupBy("kota").agg(
    spark_sum("pendapatan").alias("total_pendapatan")
)

# Join dengan df_target dan menghitung pencapaian_persen
df_hasil_a = ringkasan_kota.join(df_target, on="kota", how="inner")
df_hasil_a = df_hasil_a.withColumn(
    "pencapaian_persen",
    (col("total_pendapatan") / col("target_bulanan") * 100)
)

df_hasil_a.orderBy(col("pencapaian_persen").desc()).show()

[Stage 11:===========================================>              (3 + 1) / 4]

+----------+----------------+--------------+----------+------------------+
|      kota|total_pendapatan|target_bulanan|pic_cabang| pencapaian_persen|
+----------+----------------+--------------+----------+------------------+
| Purworejo|        45650000|      30000000|     Fitri|152.16666666666669|
|      Solo|        33475000|      40000000|      Bayu|           83.6875|
|Yogyakarta|        47275000|      60000000|      Joko| 78.79166666666667|
|  Magelang|        31650000|      45000000|      Rani| 70.33333333333334|
|  Semarang|        38175000|      55000000|      Sari|  69.4090909090909|
+----------+----------------+--------------+----------+------------------+



In [8]:
#B. Window Function - Kategori Terlaris per Kota

from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

# Ringkas pendapatan per kota dan kategori terlebih dahulu agar pencarian top-1 relevan
pendapatan_kategori = df_transaksi.groupBy("kota", "kategori").agg(
    spark_sum("pendapatan").alias("pendapatan_kategori")
)

# Mendefinisikan window: partisi per kota, urutkan dari pendapatan tertinggi
window_kota = Window.partitionBy("kota").orderBy(col("pendapatan_kategori").desc())

# Menggunakan row_number untuk mencari peringkat 1
df_hasil_b = pendapatan_kategori.withColumn("peringkat", row_number().over(window_kota))

# Memfilter hanya top 1 dan membuang kolom peringkat agar rapi
df_hasil_b.filter(col("peringkat") == 1).drop("peringkat").show()

+----------+--------------------+-------------------+
|      kota|            kategori|pendapatan_kategori|
+----------+--------------------+-------------------+
|  Magelang|Kesehatan & Kecan...|            7275000|
| Purworejo|Kesehatan & Kecan...|           10075000|
|  Semarang|        Rumah Tangga|           11125000|
|      Solo|Kesehatan & Kecan...|            8425000|
|Yogyakarta|             Fashion|           13325000|
+----------+--------------------+-------------------+



In [9]:
#C. Spark SQL

# Mendaftarkan kedua DataFrame sebagai temporary view
df_transaksi.createOrReplaceTempView("transaksi")
df_target.createOrReplaceTempView("target")

# Mengeksekusi kueri SQL
hasil_sql = spark.sql('''
    SELECT t.kota, tg.pic_cabang, COUNT(t.order_id) AS jumlah_transaksi
    FROM transaksi t
    JOIN target tg ON t.kota = tg.kota
    GROUP BY t.kota, tg.pic_cabang
    ORDER BY jumlah_transaksi DESC
''')
hasil_sql.show()

+----------+----------+----------------+
|      kota|pic_cabang|jumlah_transaksi|
+----------+----------+----------------+
| Purworejo|     Fitri|             116|
|Yogyakarta|      Joko|             110|
|      Solo|      Bayu|              95|
|  Semarang|      Sari|              93|
|  Magelang|      Rani|              86|
+----------+----------+----------------+



Berdasarkan analisis perbandingan target, cabang yang memiliki kinerja terbaik adalah Purworejo dengan pencapaian sebesar 152,16% karena berhasil mendapatkan total pendapatan hingga Rp45.650.000, melewati target bulanan mereka secara signifikan. Kesuksesan ini didorong oleh penjualan maksimal pada kategori Kesehatan & Kecantikan yang menyumbang pendapatan tertinggi di kota tersebut sebesar Rp10.075.000. Sebaliknya, cabang yang kinerjanya paling memprihatinkan dan sangat membutuhkan perhatian serius dari manajemen adalah Semarang di bawah pengawasan PIC Sari. Cabang ini menempati posisi terendah dengan pencapaian hanya 69,40%, atau baru mengumpulkan Rp38.175.000 dari beban target bulanan yang cukup tinggi yakni Rp55.000.000. Oleh karena itu, manajemen harus segera berdiskusi dengan PIC untuk mengevaluasi strategi pemasaran secara menyeluruh, menyesuaikan kembali beban target yang mungkin terlampau tinggi, serta memberikan dukungan promosi tambahan agar penjualan kategori andalan mereka, yaitu Rumah Tangga, beserta kategori lainnya dapat meningkat pesat pada periode berikutnya.